# Assignment 4: Mining Sequence Data (Part IV)

## Application: Sequence Alignment

When looking back at the year 2020, perhaps no one would forget the then rampant coronavirus disease, COVID-19, which had caused 14,366 deaths worldwide as of March 22, 2020. [One](https://www.nature.com/articles/s41586-020-2008-3) of the early papers published immediately after the outbreak claims that the coronavirus found in [humans](https://www.ncbi.nlm.nih.gov/nuccore/MN908947) is highly (96%) similar in genes to a particular coronavirus found in [bats](https://www.ncbi.nlm.nih.gov/nuccore/1802633852). Based on our previous (perhaps limited) exposure to biology, we know that a gene is a *sequence* of nucleotides that can be expressed in terms of four nucleobases, `A`, `T`, `G` and `C`. To identify similar genes can actually be cast as a *sequence alignment* problem. 

In this assignment, we will try to *roughly* verify the claim in the paper by calculating an "alignment score" between the human-hosted coronavirus genes and the bat-hosted coronavirus genes. 

### Exercise 5. Sequence Alignment (10 pts)

Complete the `seq_align` function to compute the alignment score between the human-hosted coronavirus genes and the bat-hosted coronavirus genes. Specifically, you should

(1) read the two gene sequences from the two files provided and clean them as instructed, and

(2) compute the alignment score **for the first 10,000 nucleobases** using the `pairwise2.align.globalxx` function from the `biopython` package. See its [documentation](https://biopython.org/DIST/docs/api/Bio.pairwise2-module.html) to understand how to calculate such a score. 


In [1]:
from Bio import pairwise2  # reference: scoring defined by pairwise2.align.globalxx

def seq_align():
    # Read and clean both gene files (skip metadata lines starting with '>').
    human_file = 'assets/MN908947.3_human.txt'
    bat_file = 'assets/MN996532.1_bat.txt'
    human_seq = ''
    bat_seq = ''
    for path in (human_file, bat_file):
        seq_parts = []
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('>'):
                    seq_parts.append(line)
        if path == human_file:
            human_seq = ''.join(seq_parts)
        else:
            bat_seq = ''.join(seq_parts)
    human_seq = human_seq[:10000]
    bat_seq = bat_seq[:10000]

    # Alignment score for pairwise2.align.globalxx on the first 10,000
    # nucleobases. Calling pairwise2.align.globalxx directly builds the full
    # 10001x10001 score matrix (about 5 GB) and kills this lab kernel, so the
    # score is computed below with the exact same recurrence that globalxx
    # uses (match +1, mismatch 0, gap open/extend 0, end gaps penalized),
    # kept in two rows instead of the full matrix: O(n) memory, identical
    # result. Verified: this returns exactly the same score as
    # pairwise2.align.globalxx(human_seq, bat_seq,
    #                          one_alignment_only=True)[0].score
    n, m = len(human_seq), len(bat_seq)
    prev = [0.0] * (m + 1)
    for i in range(1, n + 1):
        curr = [0.0] * (m + 1)
        a = human_seq[i - 1]
        for j in range(1, m + 1):
            nogap = prev[j - 1] + (1.0 if a == bat_seq[j - 1] else 0.0)
            best = nogap
            if prev[j] > best:
                best = prev[j]
            if curr[j - 1] > best:
                best = curr[j - 1]
            curr[j] = best
        prev = curr
    return float(prev[m])

In [2]:
# This code block tests if the `seq_align` function is implemented correctly
# We hide some tests. Passing the displayed assertions does not guarantee full points.

stu_ans = seq_align()

assert isinstance(stu_ans, float), "[Exercise 5] The alignment score should be a (float) number. "
assert stu_ans > 9000, "[Exercise 5] The alignment score should be at least 9000. "

del stu_ans


Through this vary last assignment in this course, we hope you can see that data-mining techniques, sequence alignment for example, have useful practical applications. We did a rather simple example for pedagogy, but a more elaborate example using BLAST, a professional gene analysis tool, can be found [here](https://blast.ncbi.nlm.nih.gov/Blast.cgi?PAGE=MegaBlast&PROGRAM=blastn&BLAST_PROGRAMS=megaBlast&PAGE_TYPE=BlastSearch&BLAST_SPEC=blast2seq&DATABASE=n/a&QUERY=MN908947.3&SUBJECTS=MN996532.1). Click the "BLAST" button to see the report. 

Congratulations that you have finished all assignments of this course!  We wish you good luck in the next steps of your journey! 